# Phase 1 — Exploration, Tokenization & Cleaning
**Dataset:** Amazon Fine Food Reviews (568,454 reviews, Kaggle `snap/amazon-fine-food-reviews`)
Run from the repository root after placing `Reviews.csv` in `data/raw/`.

In [1]:
import sys, re, json, random, collections
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == 'notebooks' else str(Path.cwd()))
import pandas as pd, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.load_data import load_raw
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIG = ROOT / 'figures'; FIG.mkdir(exist_ok=True)
pd.set_option('display.width', 140); pd.set_option('display.max_colwidth', 120)
df = load_raw()
print('Loaded', df.shape, 'columns:', list(df.columns))

Loaded (568454, 10) columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


## Task 2 — Exploration & basic statistics

In [2]:
df['n_words'] = df['Text'].str.split().str.len()
print(f"Number of documents          : {len(df):,}")
print(f"Average length (words)       : {df['n_words'].mean():.1f}  (median {df['n_words'].median():.0f}, max {df['n_words'].max():,})")
print(f"Missing Summary / ProfileName: {df['Summary'].isna().sum()} / {df['ProfileName'].isna().sum()}")
print(f"Distinct products / users    : {df['ProductId'].nunique():,} / {df['UserId'].nunique():,}")
print(f"Exact duplicate reviews      : {df.duplicated(['UserId','Time','Text']).sum():,}  (same user, time, text)")
print()
print('Target distribution (Score 1-5):')
dist = df['Score'].value_counts().sort_index()
print(pd.DataFrame({'count': dist, 'percent': (100*dist/len(df)).round(1)}))
sent = df['Score'].map({1:'negative',2:'negative',3:'neutral',4:'positive',5:'positive'})
print('\nDerived sentiment target:'); print(sent.value_counts().to_frame('count').assign(percent=lambda d:(100*d['count']/len(df)).round(1)))

Number of documents          : 568,454
Average length (words)       : 80.3  (median 56, max 3,432)
Missing Summary / ProfileName: 27 / 26
Distinct products / users    : 74,258 / 256,059


Exact duplicate reviews      : 174,562  (same user, time, text)

Target distribution (Score 1-5):
        count  percent
Score                 
1       52268      9.2
2       29769      5.2
3       42640      7.5
4       80655     14.2
5      363122     63.9

Derived sentiment target:
           count  percent
Score                    
positive  443777     78.1
negative   82037     14.4
neutral    42640      7.5


In [3]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
dist.plot.bar(ax=ax[0], color='#4C78A8'); ax[0].set_title('Score distribution'); ax[0].set_xlabel('Score'); ax[0].set_ylabel('reviews')
ax[1].hist(df['n_words'].clip(upper=400), bins=60, color='#F58518'); ax[1].set_title('Review length (words, clipped at 400)'); ax[1].set_xlabel('words')
plt.tight_layout(); plt.savefig(FIG/'fig1_score_and_length.png', dpi=150); print('saved fig1')

saved fig1


### Tokens vs types, Zipf's law (random sample of 50,000 reviews)

In [4]:
sample = df.sample(50_000, random_state=42)
TOK = re.compile(r"[a-z]+(?:'[a-z]+)?")          # lower-case word tokens
tokens = [t for text in sample['Text'] for t in TOK.findall(text.lower())]
freq = collections.Counter(tokens)
N, V = len(tokens), len(freq)
hapax = sum(1 for c in freq.values() if c == 1)
ranked = freq.most_common()
top100 = sum(c for _, c in ranked[:100]) / N
print(f"Tokens (N) = {N:,}   Types / vocabulary (V) = {V:,}   type/token ratio = {V/N:.4f}")
print(f"Types occurring exactly once (hapax) : {hapax:,} = {100*hapax/V:.1f}% of the vocabulary")
print(f"Top 100 types cover {100*top100:.1f}% of all tokens")
print('Top 15:', ranked[:15])

Tokens (N) = 4,030,821   Types / vocabulary (V) = 41,818   type/token ratio = 0.0104
Types occurring exactly once (hapax) : 17,183 = 41.1% of the vocabulary
Top 100 types cover 54.3% of all tokens
Top 15: [('the', 163652), ('i', 132911), ('and', 112928), ('a', 108469), ('to', 88975), ('it', 81505), ('of', 70327), ('is', 64002), ('this', 58856), ('br', 57126), ('for', 48379), ('in', 48211), ('my', 40887), ('that', 37770), ('but', 33094)]


In [5]:
ranks = np.arange(1, len(ranked)+1); counts = np.array([c for _, c in ranked])
fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.loglog(ranks, counts, color='#4C78A8', label='observed')
ax.loglog(ranks, counts[0]/ranks, '--', color='grey', label='ideal Zipf (1/rank)')
ax.set_xlabel('rank (log)'); ax.set_ylabel('frequency (log)'); ax.set_title('Zipf plot, 50k-review sample'); ax.legend()
plt.tight_layout(); plt.savefig(FIG/'fig2_zipf.png', dpi=150); print('saved fig2')

saved fig2


### Real examples of messy text

In [6]:
def first(mask, maxlen=420):
    cand = df[mask & (df['Text'].str.len() < maxlen)]
    return cand.iloc[0]
examples = {
 'HTML tags (<br />)'              : first(df['Text'].str.contains('<br />')),
 'HTML entity / quotes'            : first(df['Text'].str.contains(r'&quot;|&#\d+;|&amp;')),
 'URL / anchor tag'                : first(df['Text'].str.contains('http')),
 'Shouting + repeated punctuation' : first(df['Text'].str.contains(r'[A-Z]{4,}.*!!!')),
 'Missing space after full stop'   : first(df['Text'].str.contains(r'[a-z]{3}\.[A-Z][a-z]{2}')),
 'Non-ASCII characters'            : first(df['Text'].str.contains(r'[^\x00-\x7f]')),
}
for k, row in examples.items():
    print(f"--- {k}  (Id={row['Id']})"); print(repr(row['Text'])); print()

--- HTML tags (<br />)  (Id=22)
'I bought these for my husband who is currently overseas. He loves these, and apparently his staff likes them also.<br />There are generous amounts of Twizzlers in each 16-ounce bag, and this was well worth the price. <a href="http://www.amazon.com/gp/product/B001GVISJM">Twizzlers, Strawberry, 16-Ounce Bags (Pack of 6)</a>'

--- HTML entity / quotes  (Id=3593)
"I don't understand why this dog food is not full of reviews. Checkout [&#8230;] they give this dog food 4 out of 5 stars which beats science diet and purina one which I use to use and diamond natural is less expensive."

--- URL / anchor tag  (Id=22)
'I bought these for my husband who is currently overseas. He loves these, and apparently his staff likes them also.<br />There are generous amounts of Twizzlers in each 16-ounce bag, and this was well worth the price. <a href="http://www.amazon.com/gp/product/B001GVISJM">Twizzlers, Strawberry, 16-Ounce Bags (Pack of 6)</a>'

--- Shouting + repeated pu

## Task 3 — Tokenization & regular expressions

In [7]:
import spacy
from nltk.tokenize import TreebankWordTokenizer, TweetTokenizer
regex_tok = re.compile(r"\w+(?:'\w+)?|[^\w\s]")
tb, tw = TreebankWordTokenizer(), TweetTokenizer()
nlp_blank = spacy.blank('en')
sentences = [
 "I didn't like it, but my dog's gluten-free treats cost $3.99 (12oz) - good value!!!",
 "Great product :) 5-star, would buy again. Visit http://example.com <br />Thanks!",
 "It's not bad; I can't say it's the best I've tried, e.g. vs. Brand-X (2 lb.)",
]
for s in sentences:
    print('TEXT     :', s)
    print('regex    :', regex_tok.findall(s))
    print('NLTK Tree:', tb.tokenize(s))
    print('NLTK Tweet:', tw.tokenize(s))
    print('spaCy    :', [t.text for t in nlp_blank(s)])
    print()

TEXT     : I didn't like it, but my dog's gluten-free treats cost $3.99 (12oz) - good value!!!
regex    : ['I', "didn't", 'like', 'it', ',', 'but', 'my', "dog's", 'gluten', '-', 'free', 'treats', 'cost', '$', '3', '.', '99', '(', '12oz', ')', '-', 'good', 'value', '!', '!', '!']
NLTK Tree: ['I', 'did', "n't", 'like', 'it', ',', 'but', 'my', 'dog', "'s", 'gluten-free', 'treats', 'cost', '$', '3.99', '(', '12oz', ')', '-', 'good', 'value', '!', '!', '!']
NLTK Tweet: ['I', "didn't", 'like', 'it', ',', 'but', 'my', "dog's", 'gluten-free', 'treats', 'cost', '$', '3.99', '(', '12oz', ')', '-', 'good', 'value', '!', '!', '!']
spaCy    : ['I', 'did', "n't", 'like', 'it', ',', 'but', 'my', 'dog', "'s", 'gluten', '-', 'free', 'treats', 'cost', '$', '3.99', '(', '12', 'oz', ')', '-', 'good', 'value', '!', '!', '!']

TEXT     : Great product :) 5-star, would buy again. Visit http://example.com <br />Thanks!
regex    : ['Great', 'product', ':', ')', '5', '-', 'star', ',', 'would', 'buy', 'again', '

In [8]:
# Average number of tokens per method on 5,000 reviews, and how each handles 3 tricky phenomena
smp = df['Text'].sample(5000, random_state=1).tolist()
rows = []
for name, f in [('regex', regex_tok.findall), ('NLTK Treebank', tb.tokenize),
                ('NLTK Tweet', tw.tokenize), ('spaCy (blank en)', lambda s: [t.text for t in nlp_blank(s)])]:
    toks = [f(s) for s in smp]
    allt = [t for ts in toks for t in ts]
    rows.append({'method': name, 'tokens/review': round(len(allt)/len(smp), 1), 'types': len(set(allt))})
print(pd.DataFrame(rows).to_string(index=False))
print()
tests = {"contraction": "don't", "hyphen": "gluten-free", "price": "$3.99", "units": "12oz", "emoticon": ":)", "html": "<br />"}
print(f"{'phenomenon':12} {'regex':28} {'Treebank':26} {'Tweet':22} spaCy")
for k, v in tests.items():
    print(f"{k:12} {str(regex_tok.findall(v)):28} {str(tb.tokenize(v)):26} {str(tw.tokenize(v)):22} {[t.text for t in nlp_blank(v)]}")

          method  tokens/review  types
           regex           99.4  19236
   NLTK Treebank           94.5  24695
      NLTK Tweet           97.1  20552
spaCy (blank en)           96.4  21795

phenomenon   regex                        Treebank                   Tweet                  spaCy
contraction  ["don't"]                    ['do', "n't"]              ["don't"]              ['do', "n't"]
hyphen       ['gluten', '-', 'free']      ['gluten-free']            ['gluten-free']        ['gluten', '-', 'free']
price        ['$', '3', '.', '99']        ['$', '3.99']              ['$', '3.99']          ['$', '3.99']
units        ['12oz']                     ['12oz']                   ['12oz']               ['12', 'oz']
emoticon     [':', ')']                   [':', ')']                 [':)']                 [':)']
html         ['<', 'br', '/', '>']        ['<', 'br', '/', '>']      ['<', 'br', '/', '>']  ['<', 'br', '/', '>']


### Three (plus) regular expressions designed for this data

In [9]:
patterns = {
 'HTML tags'       : (r'<[^>]+>',                      "Matches '<', any run of non-'>' characters, then '>' → <br />, <a href=...>, </b>"),
 'URLs'            : (r'(?:https?://|www\.)\S+',        "http:// or https:// or www. followed by non-space characters"),
 'Prices'          : (r'\$\s?\d+(?:[.,]\d+)*',          "literal $, optional space, digits, optional .99 / ,000 groups → $3.99, $1,200.50"),
 'Repeated !?.'    : (r'([!?.])\1{1,}',                "a punctuation mark followed by itself 1+ times (back-reference) → !!! ... ??"),
 'Quantity + unit' : (r'\b\d+(?:\.\d+)?\s?(?:oz|lb|lbs|ct|count|pack|g|kg|ml|l)\b', "number glued to a pack-size unit → 12oz, 2 lb, 24 count"),
 'HTML entities'   : (r'&(?:#\d+|[a-z]+);',            "& then #digits or letters then ; → &quot; &#34; &amp;"),
}
rows = []
for name, (pat, expl) in patterns.items():
    rx = re.compile(pat, re.I)
    hits = df['Text'].map(lambda s: rx.search(s) is not None)
    ex = rx.search(df.loc[hits, 'Text'].iloc[0]).group(0) if hits.any() else ''
    rows.append({'pattern': name, 'regex': pat, 'reviews matching': int(hits.sum()), 'pct': round(100*hits.mean(), 1), 'example': ex})
print(pd.DataFrame(rows).to_string(index=False))
print()
for name, (pat, expl) in patterns.items(): print(f"{name:16}: {expl}")

        pattern                                                     regex  reviews matching  pct                                                 example
      HTML tags                                                   <[^>]+>            147324 25.9                                                  <br />
           URLs                                    (?:https?://|www\.)\S+             11788  2.1 http://www.amazon.com/gp/product/B001GVISJM">Twizzlers,
         Prices                                      \$\s?\d+(?:[.,]\d+)*             24308  4.3                                                   $0.30
   Repeated !?.                                             ([!?.])\1{1,}            103439 18.2                                                     ...
Quantity + unit \b\d+(?:\.\d+)?\s?(?:oz|lb|lbs|ct|count|pack|g|kg|ml|l)\b             31802  5.6                                                   45 Kg
  HTML entities                                         &(?:#\d+|[a-z]+);         

In [10]:
# Tokenization challenge 1: contractions & negation. Treebank splits "didn't" -> did + n't (good); the
# regex keeps "didn't" whole; stripping punctuation naively gives "didn" + "t" and LOSES the negation.
print('naive punct strip :', re.sub(r"[^\w\s]", "", "I didn't like it").split())
print('regex             :', regex_tok.findall("I didn't like it"))
print('Treebank          :', tb.tokenize("I didn't like it"))
# Tokenization challenge 2: hyphenation / compounds
for w in ["gluten-free", "low-carb", "5-star", "k-cups", "tastes-great-less-filling"]:
    print(f"{w:28} regex={regex_tok.findall(w)}  Treebank={tb.tokenize(w)}")
# challenge 3: missing space after full stop glues two sentences together
s = "Great taste.Would buy again.Highly recommend"
print(s, '->', regex_tok.findall(s), ' / ', tb.tokenize(s))

naive punct strip : ['I', 'didnt', 'like', 'it']
regex             : ['I', "didn't", 'like', 'it']
Treebank          : ['I', 'did', "n't", 'like', 'it']
gluten-free                  regex=['gluten', '-', 'free']  Treebank=['gluten-free']
low-carb                     regex=['low', '-', 'carb']  Treebank=['low-carb']
5-star                       regex=['5', '-', 'star']  Treebank=['5-star']
k-cups                       regex=['k', '-', 'cups']  Treebank=['k-cups']
tastes-great-less-filling    regex=['tastes', '-', 'great', '-', 'less', '-', 'filling']  Treebank=['tastes-great-less-filling']
Great taste.Would buy again.Highly recommend -> ['Great', 'taste', '.', 'Would', 'buy', 'again', '.', 'Highly', 'recommend']  /  ['Great', 'taste.Would', 'buy', 'again.Highly', 'recommend']


## Task 4 — Cleaning & normalization pipeline

In [11]:
from src import cleaning as C
from src.cleaning import clean_text, normalize_unicode, remove_noise, expand_contractions, tokenize
review = "I DIDN'T like it!!! <br />Paid $3.99 at http://x.com &quot;Gluten-Free&quot; cookies were better than the BEST ones I've tried..."
a = normalize_unicode(review);     print('1 NFC/quotes   :', a)
b = remove_noise(a);               print('2 noise removal:', b)
c = b.lower();                     print('3 case folding :', c)
d = expand_contractions(c);        print('4 contractions :', d)
e = tokenize(d);                   print('5 tokens       :', e)
f = [t for t in e if t not in C.STOP_WORDS]; print('6 stop words   :', f)
C.lemmatize_types(f); g = [C._LEMMA_CACHE[t] for t in f]; print('7 lemmas       :', g)
print('clean_text()   :', clean_text(review))

1 NFC/quotes   : I DIDN'T like it!!! <br />Paid $3.99 at http://x.com &quot;Gluten-Free&quot; cookies were better than the BEST ones I've tried...
2 noise removal: I DIDN'T like it!  Paid   at   "Gluten-Free" cookies were better than the BEST ones I've tried.
3 case folding : i didn't like it!  paid   at   "gluten-free" cookies were better than the best ones i've tried.
4 contractions : i did not like it!  paid   at   "gluten-free" cookies were better than the best ones i have tried.
5 tokens       : ['i', 'did', 'not', 'like', 'it', 'paid', 'at', 'gluten', 'free', 'cookies', 'were', 'better', 'than', 'the', 'best', 'ones', 'i', 'have', 'tried']
6 stop words   : ['not', 'like', 'paid', 'gluten', 'free', 'cookies', 'better', 'best', 'ones', 'tried']
7 lemmas       : ['not', 'like', 'pay', 'gluten', 'free', 'cookie', 'better', 'best', 'one', 'try']
clean_text()   : not like pay gluten free cookie better best one try


In [12]:
# Stop-word list check: negations must survive
from spacy.lang.en.stop_words import STOP_WORDS as SPACY_STOP
negs = ['no','not','nor','never','none','nothing','nobody','neither','cannot','without','very','too','but','only','just','really']
print('spaCy default list size :', len(SPACY_STOP), '| our list size:', len(C.STOP_WORDS))
print('negation/intensity words in spaCy default (would be deleted):', [w for w in negs if w in SPACY_STOP])
print('still removed by our list                                  :', [w for w in negs if w in C.STOP_WORDS])
print('\nBefore/after with default vs our list:')
t = "this is not good and i would never buy it again but the other one was very nice"
print('  default :', ' '.join(w for w in t.split() if w not in SPACY_STOP))
print('  ours    :', ' '.join(w for w in t.split() if w not in C.STOP_WORDS))

spaCy default list size : 326 | our list size: 291
negation/intensity words in spaCy default (would be deleted): ['no', 'not', 'nor', 'never', 'none', 'nothing', 'nobody', 'neither', 'cannot', 'without', 'very', 'too', 'but', 'only', 'just', 'really']
still removed by our list                                  : []

Before/after with default vs our list:
  default : good buy nice
  ours    : not good never buy again but very nice


In [13]:
# Stemming vs lemmatization on words from this corpus
from nltk.stem import PorterStemmer
ps = PorterStemmer()
words = ['cookies','tasting','tastes','delicious','disappointed','disappointing','bought','better','best','worse',
         'flavors','flavorful','amazingly','packaged','packaging','unsalted','happily','dogs','eating','was']
C.lemmatize_types(words)
print(pd.DataFrame({'word': words, 'Porter stem': [ps.stem(w) for w in words], 'lemma': [C.lemma(w) for w in words]}).to_string(index=False))

         word Porter stem      lemma
      cookies       cooki     cookie
      tasting        tast      taste
       tastes        tast      taste
    delicious      delici  delicious
 disappointed  disappoint disappoint
disappointing  disappoint disappoint
       bought      bought        buy
       better      better     better
         best        best       best
        worse        wors      worse
      flavors      flavor     flavor
    flavorful      flavor  flavorful
    amazingly   amazingli  amazingly
     packaged      packag    package
    packaging      packag    package
     unsalted      unsalt   unsalted
      happily     happili    happily
         dogs         dog        dog
       eating         eat        eat
          was          wa         be


In [14]:
stats = json.loads((ROOT/'data'/'processed'/'vocab_stats.json').read_text())
print(json.dumps(stats, indent=2))
# intermediate vocabularies on the same de-duplicated data for a fair comparison
dd = df.drop_duplicates(['UserId','Time','Text'])
v_raw = set(t for x in dd['Text'] for t in x.split())
v_lower = set(t for x in dd['Text'] for t in re.findall(r"[a-z]+", x.lower()))
print()
print(f"vocabulary, raw whitespace tokens (case-sensitive)    : {len(v_raw):>9,}")
print(f"vocabulary, lower-cased letter-only tokens (no clean) : {len(v_lower):>9,}")
print(f"vocabulary, after full cleaning pipeline              : {stats['vocab_after']:>9,}")
print(f"reduction vs raw: {stats['vocab_reduction_pct']}%   |   tokens {stats['tokens_before']:,} -> {stats['tokens_after']:,}")

{
  "rows_raw": 568454,
  "rows_after_dedup": 393892,
  "duplicates_removed": 174562,
  "rows_final": 393891,
  "empty_after_cleaning": 1,
  "vocab_before": 586944,
  "vocab_after": 90596,
  "tokens_before": 31445397,
  "tokens_after": 15192247,
  "vocab_reduction_pct": 84.56,
  "seconds": 68.1
}



vocabulary, raw whitespace tokens (case-sensitive)    :   586,944
vocabulary, lower-cased letter-only tokens (no clean) :   111,150
vocabulary, after full cleaning pipeline              :    90,596
reduction vs raw: 84.56%   |   tokens 31,445,397 -> 15,192,247


In [15]:
clean = pd.read_parquet(ROOT/'data'/'processed'/'cleaned_reviews.parquet')
fig, ax = plt.subplots(figsize=(6.2, 3.2))
labels = ['raw whitespace\ntokens', 'lower-case,\nletters only', 'full cleaning\npipeline']
vals = [len(v_raw), len(v_lower), stats['vocab_after']]
bars = ax.bar(labels, vals, color=['#E45756', '#F58518', '#54A24B'])
for b, v in zip(bars, vals): ax.text(b.get_x()+b.get_width()/2, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('vocabulary size'); ax.set_title('Vocabulary size before vs after cleaning'); ax.set_ylim(0, max(vals)*1.12)
plt.tight_layout(); plt.savefig(FIG/'fig3_vocab_before_after.png', dpi=150); print('saved fig3')

saved fig3


## Task 5 — Output

In [16]:
print(clean.shape); print(clean.dtypes.to_string()); print()
print(clean[['Id','Score','sentiment','n_exclaim','caps_ratio','clean_text']].head(5).to_string())
print()
for i in [0, 1]:
    print('RAW  :', clean.loc[i, 'Text'][:200]); print('CLEAN:', clean.loc[i, 'clean_text'][:200]); print()
assert clean['clean_text'].str.len().min() > 0 and clean['Score'].between(1, 5).all()
print('sanity checks passed:', len(clean), 'rows')

(393891, 14)
Id                          int64
ProductId                     str
UserId                        str
Score                       int64
sentiment                     str
HelpfulnessNumerator        int64
HelpfulnessDenominator      int64
Time                        int64
Summary                       str
Text                          str
clean_summary                 str
clean_text                    str
n_exclaim                   int64
caps_ratio                float64

   Id  Score sentiment  n_exclaim  caps_ratio                                                                                                                                                                                                                                                                           clean_text
0   1      5  positive          0      0.0236                                                                                                                            buy vitality can d